# Trend Following + Multi-Factor Strategy

Combines market regime detection (CSI 300 MA crossovers) with multi-factor stock selection:

```
data (stocks) -> multifactor -> signals
data (CSI 300) -> MarketRegime -> exposure_frac
                                    |
signals + exposure -> equal_weight(capital * exposure) -> risk -> backtest
```

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib
import pandas as pd

matplotlib.use('Agg')
from pathlib import Path

import matplotlib.pyplot as plt

from backtest.engine import BacktestEngine
from backtest.walk_forward import walk_forward_backtest
from config.loader import build_combined_strategy, load_config
from data import validate_ohlcv
from data.fetcher import fetch_daily, fetch_index_daily
from data.filters import detect_limit_price, detect_suspension
from data.storage import load_parquet, save_parquet
from data.universe import resolve_universe
from portfolio.allocator import equal_weight
from risk.position_limit import apply_position_limit
from risk.tradability import enforce_t1, filter_tradable
from strategies.builtin.market_regime import market_regime_exposure
from strategies.builtin.multifactor import multifactor_signal
from visualization.charts import plot_backtest_summary

print('All imports OK')

## Step 1: Load Config & Data

In [ ]:
cfg = load_config(Path('../configs/trend_multifactor.yaml'))
combined = build_combined_strategy(cfg)
regime = combined['regime']
strategy = combined['strategy']

universe_cfg = cfg.get('universe', {})
STOCKS = resolve_universe(universe_cfg)
START = universe_cfg.get('start_date', '2023-01-01')
END = universe_cfg.get('end_date', '2026-05-23')
RAW_DIR = Path('../data/raw')

print(f'Strategy: {strategy.name}')
print(f'Regime: MA{regime.ma_short}/{regime.ma_long}')
print(f'Exposure: {regime.exposure}')
print(f'Universe: {len(STOCKS)} stocks')

In [ ]:
# Load stock data
frames = []
for code in STOCKS:
    path = RAW_DIR / f'{code}.parquet'
    if path.exists():
        df = load_parquet(path)
    else:
        df = fetch_daily(code, START, END)
        save_parquet(df, path)
    frames.append(df)

data = pd.concat(frames, ignore_index=True)
data = detect_limit_price(data)
data = detect_suspension(data)
validate_ohlcv(data)
print(f'Stocks: {len(data)} rows, {data["code"].nunique()} codes')
print(f'Date range: {data["date"].min().date()} ~ {data["date"].max().date()}')

In [ ]:
# Load CSI 300 index data for regime detection
INDEX_CODE = '000300'
index_path = RAW_DIR / f'index_{INDEX_CODE}.parquet'

if index_path.exists():
    index_data = load_parquet(index_path)
else:
    index_data = fetch_index_daily(INDEX_CODE, START, END)
    save_parquet(index_data, index_path)

print(f'Index: {len(index_data)} rows')
print(f'Date range: {index_data["date"].min().date()} ~ {index_data["date"].max().date()}')

## Step 2: Compute Market Regime Exposure

In [ ]:
# Compute exposure from CSI 300
index_close = index_data.set_index('date')['close'].sort_index()
exposure = market_regime_exposure(index_close, ma_short=regime.ma_short, ma_long=regime.ma_long)

# Map to per-stock dates
stock_dates = sorted(data['date'].unique())
exposure_per_date = pd.Series(
    [exposure.get(d, 0.6) for d in stock_dates],
    index=pd.DatetimeIndex(stock_dates)
)

# Summary
print('Regime distribution:')
regime_counts = exposure.round(1).value_counts().sort_index()
for val, count in regime_counts.items():
    label = {1.0: 'Bullish', 0.6: 'Neutral', 0.4: 'Cautious', 0.2: 'Bearish'}.get(val, f'{val}')
    print(f'  {label} ({val}): {count} days')

print(f'\nAverage exposure: {exposure.mean():.2f}')

In [ ]:
# Plot regime
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(index_close.index, index_close.values, 'b-', linewidth=0.8)
ma_s = index_close.rolling(regime.ma_short).mean()
ma_l = index_close.rolling(regime.ma_long).mean()
ax1.plot(ma_s.index, ma_s.values, 'r-', linewidth=0.6, alpha=0.7, label=f'MA{regime.ma_short}')
ax1.plot(ma_l.index, ma_l.values, 'g-', linewidth=0.6, alpha=0.7, label=f'MA{regime.ma_long}')
ax1.set_ylabel('CSI 300')
ax1.legend()
ax1.set_title('Market Regime Detection')

ax2.fill_between(exposure.index, exposure.values, alpha=0.3, color='blue')
ax2.set_ylabel('Exposure')
ax2.set_ylim(0, 1.1)
ax2.set_xlabel('Date')

plt.tight_layout()
plt.savefig('../output/trend_multifactor_regime.png', dpi=150)
plt.close()
print('Regime chart saved')

## Step 3: Generate Multi-Factor Signals

In [ ]:
signals = strategy.generate_signal(data)

print(f'Total signals: {len(signals)}')
print(f'Buy: {(signals["signal"] == 1).sum()}')
print(f'Sell: {(signals["signal"] == -1).sum()}')
print(f'Hold: {(signals["signal"] == 0).sum()}')

## Step 4: Risk Filtering

In [ ]:
filtered = filter_tradable(data, signals)
final_signals = enforce_t1(filtered)
print(f'After risk filtering: {len(final_signals[final_signals["signal"] != 0])} active signals')

## Step 5: Portfolio Allocation with Exposure Scaling

In [ ]:
CAPITAL = 1_000_000

positions = equal_weight(
    final_signals,
    data[['date', 'code', 'close']],
    capital=CAPITAL,
    exposure=exposure_per_date,
)
positions = apply_position_limit(positions, max_weight=0.3)

print(f'Positions: {len(positions)} rows')
print(f'Dates with positions: {positions["date"].nunique()}')
print(f'Avg shares per position: {positions["shares"].mean():.0f}')

## Step 6: Backtest

In [ ]:
engine = BacktestEngine(capital=CAPITAL)
result = engine.run(positions, data[['date', 'code', 'close']])

print('=== Metrics ===')
for k, v in result['metrics'].items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

In [ ]:
fig = plot_backtest_summary(result)
fig.set_size_inches(14, 8)
fig.suptitle(f'Trend+Multifactor — {strategy.name} (MA{regime.ma_short}/{regime.ma_long})', fontsize=14)
fig.tight_layout()
fig.savefig('../output/trend_multifactor_backtest.png', dpi=150, bbox_inches='tight')
plt.close()
print('Backtest chart saved')

## Step 7: Walk-Forward Validation

In [ ]:
def signal_fn(train_data, test_data):
    """Walk-forward signal function: multifactor on test data."""
    return multifactor_signal(
        test_data,
        momentum_window=20, rsi_window=14,
        hv_window=20, vol_window=20,
        rebalance=20, top_n=5, bottom_n=3,
    )


def exposure_fn(dates):
    """Walk-forward exposure function."""
    return pd.Series(
        [exposure.get(d, 0.6) for d in dates],
        index=dates,
    )


wf_result = walk_forward_backtest(
    data, signal_fn,
    train_months=12, test_months=3,
    capital=CAPITAL, max_weight=0.3,
    exposure_fn=exposure_fn,
)

print('=== Walk-Forward Results ===')
print(wf_result.to_string(index=False))
print(f'\nPeriods: {len(wf_result)}')
print(f'Profitable: {(wf_result["total_return"] > 0).sum()}/{len(wf_result)}')
print(f'Avg annual return: {wf_result["annual_return"].mean():.4f}')
print(f'Avg Sharpe: {wf_result["sharpe_ratio"].mean():.4f}')

## Step 8: Compare With/Without Trend Filter

In [ ]:
# Run without exposure scaling (pure multifactor)
signals_bare = multifactor_signal(data, top_n=5, bottom_n=3)
signals_bare = filter_tradable(data, signals_bare)
signals_bare = enforce_t1(signals_bare)
positions_bare = equal_weight(signals_bare, data[['date', 'code', 'close']], capital=CAPITAL)
positions_bare = apply_position_limit(positions_bare, max_weight=0.3)

engine_bare = BacktestEngine(capital=CAPITAL)
result_bare = engine_bare.run(positions_bare, data[['date', 'code', 'close']])

print('=== Comparison ===')
print(f'{"Metric":<20} {"Pure Multifactor":>15} {"Trend+Multifactor":>15}')
print('-' * 52)
for k in ['total_return', 'annual_return', 'sharpe_ratio', 'max_drawdown']:
    v1 = result_bare['metrics'][k]
    v2 = result['metrics'][k]
    print(f'{k:<20} {v1:>15.4f} {v2:>15.4f}')

## Analysis

(Fill in observations after running)